# Book Scraping and Analysis
Collect, clean, convert, store and query the first five catalogue pages. Run all cells in order from the module directory. The initial inspection cells reuse the first page; the complete scraper is the authoritative dataset.

In [48]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

## 1. Inspect the website and one book

In [49]:
url = "https://books.toscrape.com/"
response = requests.get(url, timeout=15)
print(response.status_code)
response.raise_for_status()
print(response.text[:500])

200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [50]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.text)
books = soup.find_all("article", class_="product_pod")
print(len(books))
first_book = books[0]
print(first_book)


    All products | Books to Scrape - Sandbox

20
<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">Â£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


In [51]:
title = first_book.h3.a["title"]
print(title)

A Light in the Attic


In [52]:
price = first_book.find("p", class_="price_color").text
print(price)

Â£51.77


In [53]:
rating = first_book.find("p")["class"][1]
print(rating)

Three


In [54]:
book_link = first_book.h3.a["href"]
print(book_link)
detail_url = urljoin(url, book_link)
print(detail_url)

catalogue/a-light-in-the-attic_1000/index.html
https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [55]:
detail_response = requests.get(detail_url, timeout=15)
detail_response.raise_for_status()
detail_soup = BeautifulSoup(detail_response.text, "html.parser")

In [56]:
availability = detail_soup.find("p", class_="instock availability").text.strip()
print(availability)

In stock (22 available)


In [57]:
breadcrumb = detail_soup.find("ul", class_="breadcrumb")
category = breadcrumb.find_all("li")[2].text.strip()
print(category)

Poetry


In [58]:
book_data = {
    "title": title,
    "price": price,
    "rating": rating,
    "availability": availability,
    "category": category
}
print(book_data)

{'title': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three', 'availability': 'In stock (22 available)', 'category': 'Poetry'}


### Pagination
Inspect the URL pattern before collecting all records.

In [59]:
for page in range(1, 6):
    if page == 1:
        url = "https://books.toscrape.com/"
    else:
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    print(url)

https://books.toscrape.com/
https://books.toscrape.com/catalogue/page-2.html
https://books.toscrape.com/catalogue/page-3.html
https://books.toscrape.com/catalogue/page-4.html
https://books.toscrape.com/catalogue/page-5.html


## 2. Collect five pages
Missing HTML fields or failed detail requests are logged and skipped. Listing-page HTTP failures stop the run rather than silently returning an incomplete catalogue.

In [60]:
all_books = []
skipped_books = []
breadcrumbs = {}
for page in range(1, 6):
    if page == 1:
        page_url = "https://books.toscrape.com/"
    else:
        page_url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(page_url, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    for book in books:
        try:
            title = book.h3.a["title"]
            price = book.find("p", class_="price_color").text
            rating = book.find("p", class_="star-rating")["class"][1]
            detail_url = urljoin(page_url, book.h3.a["href"])
            detail_response = requests.get(detail_url, timeout=15)
            detail_response.raise_for_status()
            detail_soup = BeautifulSoup(detail_response.text, "html.parser")
            availability = detail_soup.find(
                "p", class_="instock availability"
            ).text.strip()
            breadcrumb = detail_soup.find("ul", class_="breadcrumb")
            category = breadcrumb.find_all("li")[2].text.strip()
            breadcrumbs[title] = [item.get_text(strip=True) for item in breadcrumb.find_all("li")]
            all_books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category
            })
        except (AttributeError, KeyError, IndexError, TypeError, requests.RequestException) as exc:
            skipped_books.append(str(exc))
        time.sleep(0.1)
df = pd.DataFrame(all_books)
print("Total books collected:", len(df))
print("Dataset shape:", df.shape)
df.head()
print("Skipped during extraction:", len(skipped_books))

Total books collected: 100
Dataset shape: (100, 5)
Skipped during extraction: 0


## 3. Explore and clean
Drop rows with invalid required fields rather than invent prices or ratings. Preserve raw fields for inspection.

In [61]:
# Check dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         100 non-null    object
 1   price         100 non-null    object
 2   star_rating   100 non-null    object
 3   availability  100 non-null    object
 4   category      100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB


In [62]:
df.isnull().sum()

,0
title,0
price,0
star_rating,0
availability,0
category,0


In [63]:
df.duplicated().sum()

np.int64(0)

In [64]:
df["price"].head(10)

,price
0,Â£51.77
1,Â£53.74
2,Â£50.10
3,Â£47.82
4,Â£54.23
5,Â£22.65
6,Â£33.34
7,Â£17.93
8,Â£22.60
9,Â£52.15


In [65]:
df["price_gbp"] = (
    df["price"]
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
)
df[["price", "price_gbp"]].head()

,price,price_gbp
0,Â£51.77,51.77
1,Â£53.74,53.74
2,Â£50.10,50.10
3,Â£47.82,47.82
4,Â£54.23,54.23


In [66]:
df["price_gbp"] = pd.to_numeric(df["price_gbp"], errors="coerce")
print(df["price_gbp"].dtype)
df[["price_gbp"]].head()

float64


,price_gbp
0,51.77
1,53.74
2,50.10
3,47.82
4,54.23


In [67]:
df["price_gbp"].isnull().sum()

np.int64(0)

In [68]:
df["star_rating"].value_counts()

,count
star_rating,
Three,22
One,22
Five,19
Two,19
Four,18


In [69]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}
df["rating"] = df["star_rating"].map(rating_map)
df[["rating"]].head()

,rating
0,3
1,1
2,1
3,4
4,5


In [70]:
# Recognize only known availability formats; unexpected text becomes missing.
stock_text = df["availability"].str.strip()
in_stock = stock_text.str.fullmatch(r"In stock(?:\s*\(\d+ available\))?", case=False, na=False)
out_of_stock = stock_text.str.fullmatch(r"Out of stock", case=False, na=False)
df["in_stock"] = pd.Series(pd.NA, index=df.index, dtype="boolean")
df.loc[in_stock, "in_stock"] = True
df.loc[out_of_stock, "in_stock"] = False
df[["availability", "in_stock"]].head()

,availability,in_stock
0,In stock (22 available),True
1,In stock (20 available),True
2,In stock (20 available),True
3,In stock (20 available),True
4,In stock (20 available),True


In [71]:
import numpy as np
before = len(df)
valid = (df["title"].fillna("").str.strip().ne("") &
         df["category"].fillna("").str.strip().ne("") &
         np.isfinite(df["price_gbp"]) & df["price_gbp"].ge(0) &
         df["rating"].isin(range(1, 6)) & df["in_stock"].notna())
df = df.loc[valid].drop_duplicates().reset_index(drop=True)
df["price_gbp"] = df["price_gbp"].astype(float)
df["rating"] = df["rating"].astype(int)
df["in_stock"] = df["in_stock"].astype(bool)
print("Invalid or duplicate rows removed:", before - len(df))

Invalid or duplicate rows removed: 0


In [72]:
CONVERSION_RATE = 105.50
df["price_inr"] = df["price_gbp"] * CONVERSION_RATE
df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,51.77,5461.735
1,53.74,5669.570
2,50.10,5285.550
3,47.82,5045.010
4,54.23,5721.265


In [73]:
df["price_inr"] = df["price_inr"].round(2)
df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,51.77,5461.74
1,53.74,5669.57
2,50.10,5285.55
3,47.82,5045.01
4,54.23,5721.26


### Category quality
The source label Add a comment is not a meaningful genre. Preserve those books under Unknown, without guessing their true genre.

In [74]:
df[df["category"] == "Add a comment"][["title", "category"]]

,title,category
55,The Torch Is Passed: A Harding Family Story,Add a comment
63,The Mindfulness and Acceptance Workbook for An...,Add a comment
70,The Art Forger,Add a comment
88,On a Midnight Clear,Add a comment
97,Judo: Seven Steps to Black Belt (an Introducto...,Add a comment


In [75]:
for title in df.loc[df["category"] == "Add a comment", "title"]:
    print(title, breadcrumbs[title])

The Torch Is Passed: A Harding Family Story ['Home', 'Books', 'Add a comment', 'The Torch Is Passed: A Harding Family Story']
The Mindfulness and Acceptance Workbook for Anxiety: A Guide to Breaking Free from Anxiety, Phobias, and Worry Using Acceptance and Commitment Therapy ['Home', 'Books', 'Add a comment', 'The Mindfulness and Acceptance Workbook for Anxiety: A Guide to Breaking Free from Anxiety, Phobias, and Worry Using Acceptance and Commitment Therapy']
The Art Forger ['Home', 'Books', 'Add a comment', 'The Art Forger']
On a Midnight Clear ['Home', 'Books', 'Add a comment', 'On a Midnight Clear']
Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners) ['Home', 'Books', 'Add a comment', 'Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners)']


In [76]:
df["category"] = df["category"].replace("Add a comment", "Unknown")
print("Updated category count:")
print(df["category"].value_counts().head(10))
print("\nBooks with Unknown category:", (df["category"] == "Unknown").sum())

Updated category count:
category
Sequential Art    14
Nonfiction        12
Default            9
Poetry             7
Fiction            5
Food and Drink     5
Unknown            5
Young Adult        4
History            4
Fantasy            4
Name: count, dtype: int64

Books with Unknown category: 5


In [77]:
assert len(df) >= 60, "At least 60 valid books required"
assert df.loc[df["category"] != "Unknown", "category"].nunique() >= 3
assert df[["price_gbp", "rating", "in_stock", "price_inr"]].notna().all().all()
assert df["rating"].between(1, 5).all()
assert df["price_inr"].equals((df["price_gbp"] * 105.50).round(2))
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         100 non-null    object 
 1   price         100 non-null    object 
 2   star_rating   100 non-null    object 
 3   availability  100 non-null    object 
 4   category      100 non-null    object 
 5   price_gbp     100 non-null    float64
 6   rating        100 non-null    int64  
 7   in_stock      100 non-null    bool   
 8   price_inr     100 non-null    float64
dtypes: bool(1), float64(2), int64(1), object(5)
memory usage: 6.5+ KB


,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr
0,A Light in the Attic,Â£51.77,Three,In stock (22 available),Poetry,51.77,3,True,5461.74
1,Tipping the Velvet,Â£53.74,One,In stock (20 available),Historical Fiction,53.74,1,True,5669.57
2,Soumission,Â£50.10,One,In stock (20 available),Fiction,50.10,1,True,5285.55
3,Sharp Objects,Â£47.82,Four,In stock (20 available),Mystery,47.82,4,True,5045.01
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock (20 available),History,54.23,5,True,5721.26


## 4. Create and populate SQLite
This is a dedicated generated database. Existing rows in its two tables are replaced on each run.

In [78]:
import sqlite3

In [79]:
conn = sqlite3.connect("books.db")
print("Database created successfully!")

Database created successfully!


In [80]:
conn.execute("PRAGMA foreign_keys = ON")
conn.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
""")
conn.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock BOOLEAN,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")
# Clear previous records before inserting fresh data
conn.execute("DELETE FROM books")
conn.execute("DELETE FROM categories")
conn.commit()
print("Database is ready for fresh data.")

Database is ready for fresh data.


In [81]:
categories = df["category"].unique()
for category in categories:
    conn.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,)
    )
conn.commit()
print("Total unique categories:", len(categories))

Total unique categories: 29


In [82]:
category_records = []
for category in categories:
    category_id = conn.execute("SELECT category_id FROM categories WHERE category_name = ?", (category,)).fetchone()[0]
    category_records.append({"category_id": category_id, "category_name": category})
categories_table = pd.DataFrame(category_records)
books_records = []

In [83]:
for _, row in df.iterrows():
    category_id = conn.execute(
        "SELECT category_id FROM categories WHERE category_name = ?",
        (row["category"],)
    ).fetchone()[0]
    cursor = conn.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        category_id
    ))
    books_records.append({"book_id": cursor.lastrowid, "title": row["title"], "price_gbp": row["price_gbp"], "category_id": category_id})
conn.commit()
books_table = pd.DataFrame(books_records)
print("Books inserted successfully!")

Books inserted successfully!


## 5. SQL analysis
Each query and its complete result are saved to sql_queries.sql and query_outputs/ as well as displayed in the notebook.

In [84]:
from pathlib import Path
output_dir = Path("query_outputs")
output_dir.mkdir(exist_ok=True)
Path("sql_queries.sql").write_text("", encoding="utf-8")

def run_query(name, query):
    result = pd.read_sql(query, conn)
    with open("sql_queries.sql", "a", encoding="utf-8") as file:
        file.write(f"-- {name}\n{query.strip().rstrip(';')};\n\n")
    result.to_csv(output_dir / f"{name}.csv", index=False)
    print(name)
    display(result)
    return result

In [85]:
query = """
SELECT title, price_gbp, rating FROM books WHERE price_gbp > 40
"""
result = run_query("01_price_filter", query)

01_price_filter


,title,price_gbp,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5
5,The Black Maria,52.15,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
7,Our Band Could Be Your Life: Scenes from the A...,57.25,3
8,Libertarianism for Beginners,51.33,2
9,It's Only the Himalayas,45.17,2


In [86]:
query = """
SELECT title, price_gbp FROM books ORDER BY price_gbp DESC
"""
result = run_query("02_price_order", query)

02_price_order


,title,price_gbp
0,The Death of Humanity: and the Case for Life,58.11
1,Slow States of Collapse: Poems,57.31
2,Our Band Could Be Your Life: Scenes from the A...,57.25
3,The Past Never Ends,56.50
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41
...,...,...
95,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99
96,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61
97,Princess Between Worlds (Wide-Awake Princess #5),13.34
98,In Her Wake,12.84


In [87]:
query = """
SELECT title, rating, price_gbp FROM books WHERE rating = 5 ORDER BY book_id LIMIT 5
"""
result = run_query("03_rating_limit", query)

03_rating_limit


,title,rating,price_gbp
0,Sapiens: A Brief History of Humankind,5,54.23
1,Set Me Free,5,17.46
2,Scott Pilgrim's Precious Little Life (Scott Pi...,5,52.29
3,Rip it Up and Start Again,5,35.02
4,Chase Me (Paris Nights #2),5,25.27


In [88]:
query = """
SELECT DISTINCT rating FROM books ORDER BY rating
"""
result = run_query("04_distinct_rating", query)

04_distinct_rating


,rating
0,1
1,2
2,3
3,4
4,5


In [89]:
query = """
SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 40 ORDER BY price_gbp
"""
result = run_query("05_price_range", query)

05_price_range


,title,price_gbp
0,The Inefficiency Assassin: Time Management Tac...,20.59
1,Shakespeare's Sonnets,20.66
2,In the Country We Love: My Family Divided,22.00
3,America's Cradle of Quarterbacks: Western Penn...,22.50
4,The Boys in the Boat: Nine Americans and Their...,22.60
5,The Requiem Red,22.65
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11
7,The Elephant Tree,23.82
8,Olio,23.88
9,The Mindfulness and Acceptance Workbook for An...,23.89


In [90]:
query = """
SELECT books.title, books.price_gbp, categories.category_name
FROM books JOIN categories ON books.category_id = categories.category_id
ORDER BY books.book_id
"""
result = run_query("06_join", query)
sql_result = result.copy()

06_join


,title,price_gbp,category_name
0,A Light in the Attic,51.77,Poetry
1,Tipping the Velvet,53.74,Historical Fiction
2,Soumission,50.10,Fiction
3,Sharp Objects,47.82,Mystery
4,Sapiens: A Brief History of Humankind,54.23,History
...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,19.92,Sequential Art
96,"Layered: Baking, Building, and Styling Spectac...",40.11,Food and Drink
97,Judo: Seven Steps to Black Belt (an Introducto...,53.90,Unknown
98,Join,35.67,Science Fiction


In [91]:
query = """
SELECT categories.category_name, COUNT(books.book_id) AS total_books
FROM books JOIN categories ON books.category_id = categories.category_id
GROUP BY categories.category_name ORDER BY total_books DESC
"""
result = run_query("07_category_counts", query)

07_category_counts


,category_name,total_books
0,Sequential Art,14
1,Nonfiction,12
2,Default,9
3,Poetry,7
4,Unknown,5
5,Food and Drink,5
6,Fiction,5
7,Young Adult,4
8,History,4
9,Fantasy,4


In [92]:
query = """
SELECT COUNT(*) AS total_books, ROUND(AVG(price_gbp), 2) AS average_price,
MIN(price_gbp) AS minimum_price, MAX(price_gbp) AS maximum_price FROM books
"""
result = run_query("08_price_summary", query)

08_price_summary


,total_books,average_price,minimum_price,maximum_price
0,100,34.56,10.16,58.11


## 6. Compare SQL JOIN with Pandas merge
The two DataFrames below were built in memory during insertion, not read back using SQL. Compare all rows in deterministic book ID order.

In [93]:
pandas_result = pd.merge(books_table, categories_table, on="category_id", how="inner")
pandas_result = pandas_result.sort_values("book_id")[["title", "price_gbp", "category_name"]].reset_index(drop=True)
comparison = sql_result.equals(pandas_result)
print("Do SQL JOIN and pandas merge match?", comparison)
display(pd.concat({"SQL JOIN": sql_result, "Pandas merge": pandas_result}, axis=1))
pandas_result.to_csv(output_dir / "pandas_merge.csv", index=False)
assert comparison

Do SQL JOIN and pandas merge match? True


SQL JOIN            \
                                                title price_gbp   
0                                A Light in the Attic     51.77   
1                                  Tipping the Velvet     53.74   
2                                          Soumission     50.10   
3                                       Sharp Objects     47.82   
4               Sapiens: A Brief History of Humankind     54.23   
..                                                ...       ...   
95  Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...     19.92   
96  Layered: Baking, Building, and Styling Spectac...     40.11   
97  Judo: Seven Steps to Black Belt (an Introducto...     53.90   
98                                               Join     35.67   
99          In the Country We Love: My Family Divided     22.00   

                                                             Pandas merge  \
         category_name                                              title   
0               Poetry                               A Light in the Attic   
1   Historical Fiction                                 Tipping the Velvet   
2              Fiction                                         Soumission   
3              Mystery                                      Sharp Objects   
4              History              Sapiens: A Brief History of Humankind   
..                 ...                                                ...   
95      Sequential Art  Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...   
96      Food and Drink  Layered: Baking, Building, and Styling Spectac...   
97             Unknown  Judo: Seven Steps to Black Belt (an Introducto...   
98     Science Fiction                                               Join   
99          Nonfiction          In the Country We Love: My Family Divided   

                                  
   price_gbp       category_name  
0      51.77              Poetry  
1      53.74  Historical Fiction  
2      50.10             Fiction  
3      47.82             Mystery  
4      54.23             History  
..       ...                 ...  
95     19.92      Sequential Art  
96     40.11      Food and Drink  
97     53.90             Unknown  
98     35.67     Science Fiction  
99     22.00          Nonfiction  

[100 rows x 6 columns]

## 7. Final validation

In [94]:
query = "SELECT COUNT(*) AS total_books, COUNT(DISTINCT category_id) AS total_categories FROM books"
result = run_query("09_final_counts", query)
assert int(result.iloc[0]["total_books"]) == len(df)
assert int(result.iloc[0]["total_categories"]) == df["category"].nunique()
assert conn.execute("PRAGMA foreign_key_check").fetchall() == []
assert conn.execute("SELECT COUNT(*) FROM books WHERE category_id IS NULL OR rating NOT BETWEEN 1 AND 5 OR in_stock NOT IN (0, 1)").fetchone()[0] == 0
print("All final checks passed.")
conn.close()

09_final_counts


,total_books,total_categories
0,100,29


All final checks passed.


## Conclusion

In this project, I collected book details from the first five pages of the Books to Scrape website using Requests and BeautifulSoup.

After collecting the data, I used Pandas to clean the records, convert ratings and availability into suitable data types, and convert book prices from GBP to INR using a fixed conversion rate of 105.50.

I then stored the cleaned data in two related SQLite tables and performed different SQL queries to analyze the book information. I also compared the SQL JOIN results with Pandas merge results to verify their consistency.

Finally, I performed validation checks to ensure that the data was correctly stored and that there were no foreign key violations.
